# Publish: Manifest Write

**Purpose**: Generate comprehensive manifest files for deterministic, manifest-driven data loading with validation and compatibility checks.

**Key Features**:
- Master manifest with all snapshots
- Per-snapshot manifests with checksums
- Load order specification
- Schema versioning
- Compatibility metadata
- Rollback support

**Manifest Types**:
1. **Master Manifest** (`master_manifest.json`) - Registry of all snapshots
2. **Snapshot Manifest** (`snapshot_manifest.json`) - Per-snapshot load instructions
3. **Schema Manifest** (`schema_manifest.json`) - Table schemas with versioning
4. **Compatibility Manifest** (`compatibility.json`) - Version compatibility matrix

In [0]:
import os
import json
import hashlib
from datetime import datetime
from typing import Dict, List
from pyspark.sql.types import StructType
import pandas as pd

# Configuration
CATALOG = "workspace"
WAREHOUSE_SCHEMA = "warehouse"  # Dimensions and Facts
GOLD_SCHEMA = "gold"  # Analytics marts
AUDIT_SCHEMA = "audit"  # Audit tables
VOLUME_PATH = "/Volumes/workspace/publish/snapshots"
MASTER_MANIFEST_PATH = f"{VOLUME_PATH}/master_manifest.json"

# Manifest version
MANIFEST_VERSION = "1.0"
SCHEMA_VERSION = "1.0"
COMPATIBILITY_VERSION = "1.0"

print(f"✓ Configuration loaded")
print(f"✓ Volume path: {VOLUME_PATH}")

In [0]:
# Complete table dependency graph
# Format: (table_name, schema, primary_key, dependencies, table_type)
TABLE_DEFINITIONS = [
    # Warehouse Dimensions
    ("dim_source", "warehouse", "source_sk", [], "dimension"),
    ("dim_sector", "warehouse", "sector_sk", [], "dimension"),
    ("dim_role", "warehouse", "role_sk", [], "dimension"),
    ("dim_location", "warehouse", "location_sk", [], "dimension"),
    ("dim_company", "warehouse", "company_sk", [], "dimension"),
    ("dim_skill", "warehouse", "skill_sk", [], "dimension"),
    ("dim_job", "warehouse", "job_sk", [], "dimension"),
    ("dim_company_alias", "warehouse", None, ["dim_company"], "dimension"),
    
    # Warehouse Facts
    ("fact_job_postings", "warehouse", "fact_job_posting_sk", 
     ["dim_source", "dim_sector", "dim_role", "dim_location", "dim_company", "dim_job"], 
     "fact"),
    ("fact_job_lifecycle", "warehouse", "fact_job_lifecycle_sk", ["dim_job"], "fact"),
    ("fact_salary", "warehouse", "fact_salary_sk", ["dim_job", "dim_location"], "fact"),
    ("fact_pipeline_runs", "warehouse", "fact_pipeline_run_sk", [], "fact"),
    
    # Warehouse Bridges
    ("bridge_job_skill", "warehouse", None, ["dim_job", "dim_skill"], "bridge"),
    
    # Gold Analytics Marts
    ("gold_hiring_trends", "gold", None, ["fact_job_postings"], "aggregate"),
    ("gold_sector_overview", "gold", None, ["fact_job_postings"], "aggregate"),
    ("gold_location_trends", "gold", None, ["fact_job_postings"], "aggregate"),
    ("gold_company_hiring", "gold", None, ["fact_job_postings"], "aggregate"),
    ("gold_salary_trends", "gold", None, ["fact_salary"], "aggregate"),
    ("gold_skill_demand", "gold", None, ["bridge_job_skill"], "aggregate"),
    ("gold_hospitality_companies", "gold", None, ["dim_company"], "aggregate"),
    ("gold_hospitality_hiring", "gold", None, ["fact_job_postings"], "aggregate"),
    ("gold_hospitality_skills", "gold", None, ["bridge_job_skill"], "aggregate"),
    ("gold_pipeline_health", "gold", None, ["fact_pipeline_runs"], "aggregate"),
    
    # Audit
    ("audit_pipeline_runs", "audit", None, [], "audit"),
    ("audit_dq_results", "audit", None, [], "audit"),
    ("audit_access_events", "audit", None, [], "audit"),
]

print(f"✓ Table definitions loaded: {len(TABLE_DEFINITIONS)} tables")
print(f"  Dimensions: {sum(1 for _, _, _, _, t in TABLE_DEFINITIONS if t == 'dimension')}")
print(f"  Facts: {sum(1 for _, _, _, _, t in TABLE_DEFINITIONS if t == 'fact')}")
print(f"  Bridges: {sum(1 for _, _, _, _, t in TABLE_DEFINITIONS if t == 'bridge')}")
print(f"  Gold: {sum(1 for _, _, _, _, t in TABLE_DEFINITIONS if t == 'aggregate')}")
print(f"  Audit: {sum(1 for _, _, _, _, t in TABLE_DEFINITIONS if t == 'audit')}")

In [0]:
def get_table_schema(table_name: str, schema: str) -> List[Dict]:
    """Get table schema as list of field definitions."""
    table_full_name = f"{CATALOG}.{schema}.{table_name}"
    
    try:
        df = spark.table(table_full_name)
        schema_fields = [
            {
                "name": field.name,
                "type": str(field.dataType),
                "nullable": field.nullable,
                "metadata": dict(field.metadata) if field.metadata else {}
            }
            for field in df.schema.fields
        ]
        return schema_fields
    except Exception as e:
        print(f"  ⚠️  Could not load schema for {table_name}: {str(e)}")
        return []

def get_table_stats(table_name: str, schema: str) -> Dict:
    """Get table statistics."""
    table_full_name = f"{CATALOG}.{schema}.{table_name}"
    
    try:
        df = spark.table(table_full_name)
        row_count = df.count()
        column_count = len(df.columns)
        
        # Get table properties
        table_desc = spark.sql(f"DESCRIBE EXTENDED {table_full_name}").collect()
        
        return {
            "row_count": row_count,
            "column_count": column_count,
            "exists": True
        }
    except Exception as e:
        return {
            "row_count": 0,
            "column_count": 0,
            "exists": False,
            "error": str(e)
        }

def calculate_schema_hash(schema_fields: List[Dict]) -> str:
    """Calculate hash of schema for version tracking."""
    schema_str = json.dumps(schema_fields, sort_keys=True)
    return hashlib.md5(schema_str.encode()).hexdigest()

def get_latest_snapshots(num: int = 10) -> List[str]:
    """Get list of latest snapshot directories."""
    try:
        snapshots = [
            f.name for f in dbutils.fs.ls(VOLUME_PATH)
            if f.isDir() and f.name != '/' and not f.name.startswith('.')
        ]
        # Sort by name (timestamp format YYYY-MM-DD_HHmmss ensures chronological order)
        snapshots.sort(reverse=True)
        return snapshots[:num]
    except Exception as e:
        print(f"⚠️  No snapshots found: {str(e)}")
        return []

print("✓ Utility functions loaded")

In [0]:
# Generate schema manifest with versioning
print("\n" + "="*80)
print("GENERATING SCHEMA MANIFEST")
print("="*80)

schema_manifest = {
    "manifest_version": MANIFEST_VERSION,
    "schema_version": SCHEMA_VERSION,
    "generated_at": datetime.now().isoformat(),
    "catalog": CATALOG,
    "tables": []
}

for table_name, table_schema, primary_key, dependencies, table_type in TABLE_DEFINITIONS:
    print(f"\nProcessing: {table_schema}.{table_name}")
    
    # Get schema and stats
    schema_fields = get_table_schema(table_name, table_schema)
    stats = get_table_stats(table_name, table_schema)
    
    if not stats['exists']:
        print(f"  ⚠️  Table does not exist, skipping...")
        continue
    
    # Calculate schema hash for versioning
    schema_hash = calculate_schema_hash(schema_fields)
    
    table_info = {
        "name": table_name,
        "schema": table_schema,
        "table_type": table_type,
        "primary_key": primary_key,
        "dependencies": dependencies,
        "schema_hash": schema_hash,
        "schema_fields": schema_fields,
        "statistics": {
            "row_count": stats['row_count'],
            "column_count": stats['column_count']
        },
        "created_at": datetime.now().isoformat()
    }
    
    schema_manifest['tables'].append(table_info)
    print(f"  ✓ Schema hash: {schema_hash}")
    print(f"  ✓ Rows: {stats['row_count']:,}, Columns: {stats['column_count']}")

# Write schema manifest to latest snapshot and root
latest_snapshots = get_latest_snapshots(1)
if latest_snapshots:
    latest_snapshot_dir = f"{VOLUME_PATH}/{latest_snapshots[0]}"
    schema_manifest_file = f"{latest_snapshot_dir}/schema_manifest.json"
    
    with open(schema_manifest_file.replace('/Volumes/', '/dbfs/Volumes/'), 'w') as f:
        json.dump(schema_manifest, f, indent=2)
    
    print(f"\n✓ Schema manifest written: {schema_manifest_file}")
else:
    print("\n⚠️  No snapshot directory found")

# Also write to root for easy access
root_schema_manifest = f"{VOLUME_PATH}/schema_manifest.json"
with open(root_schema_manifest.replace('/Volumes/', '/dbfs/Volumes/'), 'w') as f:
    json.dump(schema_manifest, f, indent=2)

print(f"✓ Root schema manifest: {root_schema_manifest}")

In [0]:
# Generate compatibility manifest
print("\n" + "="*80)
print("GENERATING COMPATIBILITY MANIFEST")
print("="*80)

compatibility_manifest = {
    "manifest_version": MANIFEST_VERSION,
    "compatibility_version": COMPATIBILITY_VERSION,
    "generated_at": datetime.now().isoformat(),
    
    "format_compatibility": {
        "csv_version": "1.0",
        "compression": ["gzip", "none"],
        "encoding": ["utf-8"],
        "delimiter": ",",
        "quote_char": '"',
        "escape_char": '"',
        "header_required": True
    },
    
    "consumer_requirements": {
        "min_python_version": "3.8",
        "required_packages": [
            "pandas>=1.3.0",
            "sqlalchemy>=1.4.0"
        ],
        "optional_packages": [
            "psycopg2>=2.9.0"
        ]
    },
    
    "validation_rules": {
        "checksum_algorithm": "md5",
        "checksum_required": True,
        "row_count_validation": True,
        "schema_validation": True
    },
    
    "load_constraints": {
        "respect_load_order": True,
        "dependency_validation": True,
        "allow_partial_load": False,
        "require_manifest": True
    },
    
    "version_history": [
        {
            "version": "1.0",
            "released_at": "2026-06-04",
            "changes": ["Initial release"],
            "breaking_changes": []
        }
    ]
}

# Write to latest snapshot and root
if latest_snapshots:
    compat_file = f"{latest_snapshot_dir}/compatibility.json"
    with open(compat_file.replace('/Volumes/', '/dbfs/Volumes/'), 'w') as f:
        json.dump(compatibility_manifest, f, indent=2)
    print(f"✓ Compatibility manifest: {compat_file}")

root_compat_file = f"{VOLUME_PATH}/compatibility.json"
with open(root_compat_file.replace('/Volumes/', '/dbfs/Volumes/'), 'w') as f:
    json.dump(compatibility_manifest, f, indent=2)

print(f"✓ Root compatibility manifest: {root_compat_file}")

In [0]:
# Generate master manifest (registry of all snapshots)
print("\n" + "="*80)
print("GENERATING MASTER MANIFEST")
print("="*80)

# Get all snapshots
all_snapshots = get_latest_snapshots(100)
print(f"Found {len(all_snapshots)} snapshots")

snapshot_registry = []

for snapshot_id in all_snapshots:
    snapshot_dir = f"{VOLUME_PATH}/{snapshot_id}"
    manifest_file = f"{snapshot_dir}/snapshot_manifest.json"
    
    try:
        # Read snapshot manifest
        with open(manifest_file.replace('/Volumes/', '/dbfs/Volumes/'), 'r') as f:
            snapshot_data = json.load(f)
        
        # Extract key info
        snapshot_info = {
            "snapshot_id": snapshot_id,
            "path": snapshot_dir,
            "created_at": snapshot_data.get('created_at'),
            "table_count": len(snapshot_data.get('load_order', [])),
            "total_rows": snapshot_data.get('validation', {}).get('total_expected_rows', 0),
            "manifest_version": snapshot_data.get('manifest_version'),
            "status": "available"
        }
        
        snapshot_registry.append(snapshot_info)
        print(f"  ✓ {snapshot_id}: {snapshot_info['table_count']} tables, {snapshot_info['total_rows']:,} rows")
        
    except Exception as e:
        print(f"  ⚠️  Could not read manifest for {snapshot_id}: {str(e)}")
        snapshot_registry.append({
            "snapshot_id": snapshot_id,
            "path": snapshot_dir,
            "status": "incomplete",
            "error": str(e)
        })

# Build master manifest
master_manifest = {
    "manifest_version": MANIFEST_VERSION,
    "generated_at": datetime.now().isoformat(),
    "catalog": CATALOG,
    "snapshot_count": len(all_snapshots),
    "latest_snapshot": all_snapshots[0] if all_snapshots else None,
    "snapshots": snapshot_registry,
    "metadata": {
        "volume_path": VOLUME_PATH,
        "schema_version": SCHEMA_VERSION,
        "compatibility_version": COMPATIBILITY_VERSION
    }
}

# Write master manifest
with open(MASTER_MANIFEST_PATH.replace('/Volumes/', '/dbfs/Volumes/'), 'w') as f:
    json.dump(master_manifest, f, indent=2)

print(f"\n✓ Master manifest written: {MASTER_MANIFEST_PATH}")
print(f"  Total snapshots: {len(all_snapshots)}")
print(f"  Latest: {all_snapshots[0] if all_snapshots else 'None'}")

In [0]:
# Generate human-readable load order documentation
print("\n" + "="*80)
print("GENERATING LOAD ORDER DOCUMENTATION")
print("="*80)

load_order_doc = {
    "title": "Table Load Order and Dependencies",
    "generated_at": datetime.now().isoformat(),
    "description": "This document specifies the order in which tables must be loaded to respect foreign key dependencies.",
    
    "load_sequence": [
        {
            "sequence": idx + 1,
            "table": table_name,
            "type": table_type,
            "primary_key": primary_key,
            "dependencies": dependencies,
            "load_instruction": (
                f"Load {table_name} first (no dependencies)" 
                if not dependencies 
                else f"Load {table_name} after: {', '.join(dependencies)}"
            )
        }
        for idx, (table_name, table_schema, primary_key, dependencies, table_type) in enumerate(TABLE_DEFINITIONS)
    ],
    
    "dependency_graph": {
        table_name: dependencies
        for table_name, _, _, dependencies, _ in TABLE_DEFINITIONS
    },
    
    "loading_guidelines": [
        "Always load dimension tables before fact tables",
        "Always load fact tables before aggregate/gold tables",
        "Audit tables can be loaded independently",
        "Validate checksums before loading each table",
        "Validate row counts after loading each table",
        "Use the snapshot_manifest.json for automated loading"
    ]
}

if latest_snapshots:
    load_order_file = f"{latest_snapshot_dir}/load_order.json"
    with open(load_order_file.replace('/Volumes/', '/dbfs/Volumes/'), 'w') as f:
        json.dump(load_order_doc, f, indent=2)
    print(f"✓ Load order documentation: {load_order_file}")

root_load_order_file = f"{VOLUME_PATH}/load_order.json"
with open(root_load_order_file.replace('/Volumes/', '/dbfs/Volumes/'), 'w') as f:
    json.dump(load_order_doc, f, indent=2)

print(f"✓ Root load order documentation: {root_load_order_file}")

In [0]:
# Generate summary report
print("\n" + "="*80)
print("MANIFEST GENERATION SUMMARY")
print("="*80)

print(f"\n✓ Master Manifest: {MASTER_MANIFEST_PATH}")
print(f"  Snapshots tracked: {len(all_snapshots)}")
print(f"  Latest snapshot: {all_snapshots[0] if all_snapshots else 'None'}")

print(f"\n✓ Schema Manifest: {root_schema_manifest}")
print(f"  Tables documented: {len(schema_manifest['tables'])}")
print(f"  Schema version: {SCHEMA_VERSION}")

print(f"\n✓ Compatibility Manifest: {root_compat_file}")
print(f"  Format version: {compatibility_manifest['format_compatibility']['csv_version']}")
print(f"  Min Python: {compatibility_manifest['consumer_requirements']['min_python_version']}")

print(f"\n✓ Load Order Documentation: {root_load_order_file}")
print(f"  Load sequence: {len(load_order_doc['load_sequence'])} tables")

if latest_snapshots:
    print(f"\n✓ Latest Snapshot Files:")
    print(f"  {latest_snapshot_dir}/schema_manifest.json")
    print(f"  {latest_snapshot_dir}/compatibility.json")
    print(f"  {latest_snapshot_dir}/load_order.json")

print("\n" + "="*80)
print("MANIFEST GENERATION COMPLETE")
print("="*80)

# Log to audit table
audit_record = spark.createDataFrame([{
    "operation": "manifest_generation",
    "master_manifest_path": MASTER_MANIFEST_PATH,
    "snapshot_count": len(all_snapshots),
    "table_count": len(schema_manifest['tables']),
    "manifest_version": MANIFEST_VERSION,
    "generated_at": datetime.now().isoformat()
}])

audit_record.write.mode("append").saveAsTable(f"{CATALOG}.{AUDIT_SCHEMA}.publish_manifest_log")

print(f"\n✓ Logged to {AUDIT_SCHEMA}.publish_manifest_log")